# 01 · Data cleaning

**Goal:** turn the raw Ombudsman decisions into one tidy table that every later notebook uses.

**Input:** `data/raw/fos_banking_decisions.csv` (960 final decisions, banking/credit/mortgage sector, Sep 2024 – Aug 2026)
**Output:** `data/processed/decisions_clean.csv`

Steps:
1. Load and inspect the raw data
2. Check for missing values and duplicates
3. Fix inconsistent business names
4. Map each firm to its banking group and firm type
5. Label each complaint with a theme (rule-based, from `src/labels.py`)
6. Add helper columns (upheld flag, period, text length) and save

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Make "src" importable when the notebook runs from the notebooks/ folder
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

FIG_DIR = ROOT / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams.update({"figure.dpi": 110, "axes.titleweight": "bold"})
pd.set_option("display.max_colwidth", 120)

## 1. Load and inspect

In [2]:
raw = pd.read_csv(ROOT / "data" / "raw" / "fos_banking_decisions.csv", parse_dates=["decision_date"])
print(raw.shape)
raw.head()

(960, 7)


,reference,decision_date,outcome,business,sector,complaint_text,pdf_url
0,DRN-4577848,2024-10-31,Not upheld,Shawbrook Bank Limited,Consumer Credit,"Miss K’s complaint is, in essence, that Shawbrook Bank Limited (the ‘Lender’) acted unfairly and unreasonably by (1)...",https://www.financial-ombudsman.org.uk/decision/DRN-4577848.pdf
1,DRN-5077897,2024-10-31,Upheld,First Response Finance Ltd,Consumer Credit,Mr U complains that the car he acquired financed through a hire purchase agreement with First Response Finance Ltd w...,https://www.financial-ombudsman.org.uk/decision/DRN-5077897.pdf
2,DRN-4793723,2024-10-31,Not upheld,EE Limited,Consumer Credit,Mrs H has complained about the quality of a device she bought using a fixed sum loan agreement with EE Limited.,https://www.financial-ombudsman.org.uk/decision/DRN-4793723.pdf
3,DRN-4973418,2024-10-31,Upheld,Bank of Scotland Plc,Mortgages,Mr A complains about how Bank of Scotland plc trading as Halifax has dealt with him following the ending of the term...,https://www.financial-ombudsman.org.uk/decision/DRN-4973418.pdf
4,DRN-4831634,2024-10-31,Not upheld,Nationwide Building Society,Banking and Payments,Mr W complains that Nationwide Building Society did not pursue a chargeback in respect of a payment which was made t...,https://www.financial-ombudsman.org.uk/decision/DRN-4831634.pdf


In [3]:
raw.info()

<class 'pandas.DataFrame'>
RangeIndex: 960 entries, 0 to 959
Data columns (total 7 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   reference       960 non-null    str           
 1   decision_date   960 non-null    datetime64[us]
 2   outcome         960 non-null    str           
 3   business        960 non-null    str           
 4   sector          960 non-null    str           
 5   complaint_text  960 non-null    str           
 6   pdf_url         960 non-null    str           
dtypes: datetime64[us](1), str(6)
memory usage: 52.6 KB


## 2. Missing values and duplicates
Every decision has a unique reference number (DRN), so duplicates would show up as repeated references.

In [4]:
print("Missing values per column:")
print(raw.isna().sum())
print("\nDuplicate references:", raw["reference"].duplicated().sum())

Missing values per column:
reference         0
decision_date     0
outcome           0
business          0
sector            0
complaint_text    0
pdf_url           0
dtype: int64

Duplicate references: 0


## 3. Fix inconsistent business names
The same firm sometimes appears in different formats, for example `Monzo Bank Ltd` and `MONZO BANK LIMITED`.
If we don't fix this, Monzo would be split into two firms and every count by firm would be wrong.

In [5]:
from src.labels import clean_business_name, firm_group, firm_type, assign_theme, THEME_ORDER

df = raw.copy()
df["business_clean"] = df["business"].map(clean_business_name)

changed = df.loc[df["business"] != df["business_clean"], ["business", "business_clean"]].drop_duplicates()
changed

,business,business_clean
12,American Express Services Europe Limited (AESEL),American Express Services Europe Limited
64,Creation Consumer Finance Ltd,Creation Consumer Finance Limited
78,Advantage Finance Ltd,Advantage Finance Limited
102,BMW Financial Services(GB) Limited,BMW Financial Services (GB) Limited
107,Blue Motor Finance Ltd,Blue Motor Finance Limited
248,NATIONAL WESTMINSTER BANK PUBLIC LIMITED COMPANY,National Westminster Bank Plc
564,MONEYBARN NO.1 LIMITED,Moneybarn No. 1 Limited
694,MARKS AND SPENCER FINANCIAL SERVICES PLC,Marks & Spencer Financial Services plc
748,MONZO BANK LIMITED,Monzo Bank Ltd


## 4. Banking group and firm type
Customers think in terms of brands, not legal entities. Halifax decisions are published under **Bank of Scotland Plc**,
Barclays Partner Finance under **Clydesdale Financial Services Limited**, and so on. `firm_group` rolls these up.

`firm_type` splits firms into four types, which is useful because digital banks, high-street banks and car-finance
lenders get very different kinds of complaints.

In [6]:
df["firm_group"] = df["business_clean"].map(firm_group)
df["firm_type"] = df["firm_group"].map(firm_type)
df["firm_type"].value_counts()

firm_type
High-street bank               410
Other lender / payment firm    283
Digital bank / e-money         139
Car & consumer finance         128
Name: count, dtype: int64

## 5. Complaint theme (rule-based)
Each complaint's opening sentence is matched against keyword rules in `src/labels.py`. The **first** rule that matches wins,
so the order is deliberate (e.g. "fraud marker" is an account-closure issue, not a scam).

Why rules first? They're transparent and easy to explain. In notebook 03 we compare them with an unsupervised ML approach (clustering).

In [7]:
df["theme"] = df["complaint_text"].map(assign_theme)
df["theme"].value_counts()

theme
Scam / APP fraud                     232
Service & administration             203
Car finance - vehicle quality         85
Refund claim (s75 / chargeback)       83
Irresponsible lending                 83
Account closure / fraud marker        62
Timeshare / unfair credit (s140A)     59
Unauthorised transactions             49
Credit file / debt / defaults         38
Mis-selling, fees & charges           37
Mortgage                              29
Name: count, dtype: int64

In [8]:
# Spot-check: a few random examples per theme
for theme in THEME_ORDER[:4]:
    print(f"--- {theme}")
    for t in df.loc[df["theme"] == theme, "complaint_text"].sample(3, random_state=0):
        print("  •", t[:130])

--- Account closure / fraud marker
  • Miss N complains that TSB blocked and then closed her account.
  • Miss B complains Barclays Bank UK PLC (“Barclays”) closed her account without explanation.
  • Miss G complains Monzo Bank Ltd unfairly loaded a Credit Industry Fraud Avoidance System (‘CIFAS’- the UK’s fraud alert service) m
--- Timeshare / unfair credit (s140A)
  • Ms J’s complaint is, in essence, that Clydesdale Financial Services Limited, trading as Barclays Partner Finance (the ‘Lender’), a
  • Miss I complaint is, in essence, that Shawbrook Bank Limited (the ‘Lender’) acted unfairly and unreasonably by (1) being party to 
  • Miss O and Mrs R’s complaint is, in essence, that First Holiday Finance Ltd (the ‘Lender’) acted unfairly and unreasonably by (1) 
--- Mortgage
  • Mr and Mrs J complain that Santander UK Plc is charging interest on the arrears on their mortgage as they said it had previously p
  • Mr and Mrs Y complain that Intrum Mortgages UK Finance Limited has charge

## 6. Helper columns and save
- `upheld`: 1 if the Ombudsman found in the customer's favour, else 0 (our ML target later)
- `period`: the two-month window the decision falls in. The data was collected as a fixed sample of 80 decisions per window, so we compare **shares and rates** by period, never raw counts.
- `text_length`: number of words in the complaint text

In [9]:
df["upheld"] = (df["outcome"] == "Upheld").astype(int)

# Two-month windows: Sep-Oct 2024, Nov-Dec 2024, ...
start_month = (df["decision_date"].dt.month - 1) // 2 * 2 + 1
df["period_start"] = pd.to_datetime(dict(year=df["decision_date"].dt.year, month=start_month, day=1))
df["period"] = df["period_start"].dt.strftime("%b %Y") + " – " + (df["period_start"] + pd.DateOffset(months=1)).dt.strftime("%b %Y")

df["text_length"] = df["complaint_text"].str.split().str.len()

cols = ["reference", "decision_date", "period_start", "period", "outcome", "upheld", "business", "business_clean",
        "firm_group", "firm_type", "sector", "theme", "complaint_text", "text_length", "pdf_url"]
df = df[cols].sort_values("decision_date").reset_index(drop=True)

out = ROOT / "data" / "processed" / "decisions_clean.csv"
out.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(out, index=False)
print(f"Saved {len(df)} rows to {out.relative_to(ROOT)}")
df.groupby("period_start")["reference"].count().rename("decisions per window")

Saved 960 rows to data/processed/decisions_clean.csv


period_start
2024-09-01    80
2024-11-01    80
2025-01-01    80
2025-03-01    80
2025-05-01    80
2025-07-01    80
2025-09-01    80
2025-11-01    80
2026-01-01    80
2026-03-01    80
2026-05-01    80
2026-07-01    80
Name: decisions per window, dtype: int64

## What I learned in this step
- The raw data was already free of missing values and duplicates, but **business names were inconsistent**. Always check categorical columns for spelling variants before counting.
- Legal entity names hide the brand the customer used (Halifax → Bank of Scotland). A mapping table makes the analysis readable for a business audience.
- A fixed sample per period means counts over time are meaningless; I built `period` so later notebooks compare rates instead.